# NSCG 2023 + O*NET: Cleaning

Input: `data/nscg2023_onet_merged.csv`, built by `merge_nscg_onet.py` (79,800 employed respondents).

NSCG imputes most missing answers before release, so cleaning is mostly sentinel codes, one structural skip, and recoding.

In [29]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../../data")  # notebook runs from notebooks/NSCG/

# O*NET profile columns are numeric; everything from NSCG is read as text so codes
# keep their exact form ("0170", "L")
VALUES = ["Achievement", "Independence", "Recognition", "Relationships", "Support", "Working Conditions"]
ONET_NUMERIC = VALUES + ["job_zone", "n_soc", "n_onet", "n_onet_with_wv", "n_onet_with_jz"]

header = pd.read_csv(DATA / "nscg2023_onet_merged.csv", nrows=0).columns
df = pd.read_csv(
    DATA / "nscg2023_onet_merged.csv",
    dtype={c: str for c in header if c not in ONET_NUMERIC},
    keep_default_na=False,
    na_values={c: [""] for c in ONET_NUMERIC},
)
df["WTSURVY"] = df["WTSURVY"].astype(float)
print(df.shape)

(79800, 66)


/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/1339035697.py:19: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["WTSURVY"] = df["WTSURVY"].astype(float)


## 1. Drop columns that carry no information

- `LFSTAT` is 1 for everyone after the merge script's employed filter.
- `WKSYR` (works 52 weeks a year) repeats `WKSWK`: it's 1 exactly when `WKSWK` is 52.

In [30]:
df = df.drop(columns=["LFSTAT", "WKSYR"])
print(df.shape)

(79800, 64)


## 2. Turn true missing codes into blanks

From the codebook (`Ppcg23.sas`):

| Column | Code | Meaning |
|---|---|---|
| `EARN` | 9999998 | Logical skip |
| `INDCODE` | 9990 | Blank, refused, classified, uncodeable, suppressed |

In [31]:
MISSING_CODES = {"EARN": ["9999998"], "INDCODE": ["9990"]}

for col, codes in MISSING_CODES.items():
    mask = df[col].isin(codes)
    print(f"{col}: {mask.sum()} rows set to missing")
    df.loc[mask, col] = np.nan

EARN: 99 rows set to missing
INDCODE: 499 rows set to missing


## 3. Recode the telework skip

`TELEFR` (how often someone teleworks) is `L` (logical skip) for people whose `TELEC` answer says they didn't telework:

- 1: telework didn't make sense for the job
- 2: not allowed or required to telework
- 5: allowed to telework, and didn't

So `L` means "no telework", not missing. Recode it to 0 so `TELEFR` is a 0–4 frequency scale.

In [32]:
skip = df.TELEFR == "L"

df.loc[skip, "TELEFR"] = "0"
print(f"TELEFR: {skip.sum()} 'L' recoded to 0")
df.TELEFR.value_counts().sort_index()

TELEFR: 35474 'L' recoded to 0


TELEFR
0    35474
1     5975
2    11828
3    10524
4    15999
Name: count, dtype: int64

## 4. Add the target

Binary: `very_satisfied` is 1 when `JOBSATIS` is 1 (Very satisfied), and 0 for Somewhat satisfied (2), Somewhat dissatisfied (3) and Very dissatisfied (4).

In [33]:
df["very_satisfied"] = (df.JOBSATIS == "1").astype(int)

pd.DataFrame(
    {
        "n": df.very_satisfied.value_counts(),
        "pct": (df.very_satisfied.value_counts(normalize=True) * 100).round(1),
        "weighted_pct": (df.groupby("very_satisfied").WTSURVY.sum() / df.WTSURVY.sum() * 100).round(1),
    }
)

/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/2444001523.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["very_satisfied"] = (df.JOBSATIS == "1").astype(int)


,n,pct,weighted_pct
very_satisfied,,,
0,44742,56.1,55.9
1,35058,43.9,44.1


## 5. Recode for modeling

### 5a. Y/N columns to True/False

| Column | Codebook description |
|---|---|
| `MGRNAT` | Job required bachelor's-level expertise: engineering, computer science, math, natural sciences |
| `MGROTH` | Job required bachelor's-level expertise: other fields (e.g. health, business) |
| `MGRSOC` | Job required bachelor's-level expertise: social sciences |
| `JOBINS`, `JOBPENS`, `JOBPROFT`, `JOBVAC` | Available benefits: health insurance / pension or retirement plan / profit-sharing / paid vacation, sick or personal days |
| `HISPANIC` | Hispanic origin |

In [34]:
YES_NO = ["MGRNAT", "MGROTH", "MGRSOC", "JOBINS", "JOBPENS", "JOBPROFT", "JOBVAC", "HISPANIC"]

for col in YES_NO:
    df[col] = df[col].map({"Y": True, "N": False})

# Any value other than Y/N would have become NaN above
print(df[YES_NO].isna().sum().sum(), "values not Y/N")
df[YES_NO].mean().mul(100).round(1).rename("% True")

0 values not Y/N


/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/1270994525.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col] = df[col].map({"Y": True, "N": False})
/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/1

MGRNAT      55.4
MGROTH      31.8
MGRSOC      20.0
JOBINS      86.2
JOBPENS     78.1
JOBPROFT    24.9
JOBVAC      86.7
HISPANIC    13.7
Name: % True, dtype: float64

### 5b. Reverse the satisfaction and importance scales

In the codebook, 1 is the top of the scale (Very satisfied / Very important) and 4 the bottom. O*NET Work Values run the other way (higher = the job offers more), so comparing the two would flip signs. Reversing with `5 - code` makes higher mean more for both:

| New value | `SAT*` (satisfaction with...) | `FAC*` (importance of...) |
|---|---|---|
| 4 | Very satisfied | Very important |
| 3 | Somewhat satisfied | Somewhat important |
| 2 | Somewhat dissatisfied | Somewhat unimportant |
| 1 | Very dissatisfied | Not important at all |

The nine factors are the same on both sides: advancement (`ADV`), benefits (`BEN`), challenge (`CHAL`), independence (`IND`), location (`LOC`), responsibility (`RESP`), salary (`SAL`), security (`SEC`) and contribution to society (`SOC`).

`JOBSATIS` keeps its codebook coding; the target `very_satisfied` already comes from it.

In [35]:
FACTORS = ["ADV", "BEN", "CHAL", "IND", "LOC", "RESP", "SAL", "SEC", "SOC"]
SAT_COLS = ["SAT" + f for f in FACTORS]
FAC_COLS = ["FAC" + f for f in FACTORS]

before = df.SATSAL.copy()
for col in SAT_COLS + FAC_COLS:
    df[col] = 5 - df[col].astype(int)

# Spot check: codebook 1 (Very satisfied) should now be 4
pd.crosstab(before.rename("SATSAL codebook"), df.SATSAL.rename("SATSAL reversed"))

/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/2875234170.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col] = 5 - df[col].astype(int)
/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/2875234170.py:

SATSAL reversed,1,2,3,4
SATSAL codebook,,,,
1,0,0,0,22712
2,0,0,39833,0
3,0,12095,0,0
4,5160,0,0,0


### 5c. Label the unordered categories

These columns use numbers as category names (in `RACETHM`, 5 isn't "more" than 1). Converting them to labeled pandas categories keeps a model from reading the codes as amounts, so they get one-hot encoded instead, and makes EDA tables readable. Labels are shortened from the codebook (`Ppcg23.sas`).

| Column | Meaning |
|---|---|
| `NEDTP` | Employer type |
| `RACETHM` | Race/ethnicity (all groups except Hispanic are non-Hispanic) |
| `CTZN` | Citizenship |
| `MARSTA` | Marital status |
| `NDGMEMG` | Field of highest degree (major group) |
| `TELEC` | Telework situation |
| `SEX_2023` | Sex |

Ordered codes (`EMSIZE` employer size, `OCEDRLP` job-degree relatedness, `TELEFR` telework days) stay numeric, since their order means something.

In [36]:
LABELS = {
    "NEDTP": {
        "1": "Self-employed, not incorporated",
        "2": "Self-employed, incorporated",
        "3": "Private for-profit",
        "4": "Private not-for-profit",
        "5": "Local government",
        "6": "State government",
        "7": "U.S. military",
        "8": "U.S. federal government (civilian)",
        "9": "Other employer type",
    },
    "RACETHM": {
        "1": "Asian",
        "2": "American Indian/Alaska Native",
        "3": "Black",
        "4": "Hispanic, any race",
        "5": "White",
        "6": "Native Hawaiian/Pacific Islander",
        "7": "Multiple race",
    },
    "CTZN": {
        "1": "U.S. citizen, native",
        "2": "U.S. citizen, naturalized",
        "3": "Permanent resident",
        "4": "Temporary resident",
    },
    "MARSTA": {
        "1": "Married",
        "2": "Marriage-like relationship",
        "3": "Widowed",
        "4": "Separated",
        "5": "Divorced",
        "6": "Never married",
    },
    "NDGMEMG": {
        "1": "Computer and mathematical sciences",
        "2": "Life sciences",
        "3": "Physical sciences",
        "4": "Social sciences",
        "5": "Engineering",
        "6": "S&E-related fields",
        "7": "Non-S&E fields",
    },
    "TELEC": {
        "1": "Telework didn't make sense for job",
        "2": "Not allowed or required to telework",
        "3": "Required to telework",
        "4": "Allowed to telework, and did",
        "5": "Allowed to telework, and didn't",
    },
    "SEX_2023": {"F": "Female", "M": "Male"},
}

for col, labels in LABELS.items():
    # Categories keep codebook order, so tables and plots list them consistently
    df[col] = pd.Categorical(df[col].map(labels), categories=list(labels.values()))

# A code missing from LABELS would have become NaN above
print(df[list(LABELS)].isna().sum().sum(), "values without a label")
df.NEDTP.value_counts(sort=False)

0 values without a label


/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/696069118.py:57: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col] = pd.Categorical(df[col].map(labels), categories=list(labels.values()))
/var/folders/ny/p5hb2xbx46g25_c_0

NEDTP
Self-employed, not incorporated        3134
Self-employed, incorporated            5973
Private for-profit                    39346
Private not-for-profit                10529
Local government                       7619
State government                       8503
U.S. military                           378
U.S. federal government (civilian)     4008
Other employer type                     310
Name: count, dtype: int64

## 6. Flag salary, earnings and hours outliers

Nothing is dropped; each issue gets a flag so analysis can include or exclude it.

| Flag | Rule | Why |
|---|---|---|
| `salary_zero` | `SALARY` = 0 | Mostly self-employed people, who report earnings (`EARN`) but no salary |
| `salary_low` | 0 < `SALARY` < 1,000 | Implausibly low for a year of work; possibly part-time or a reporting error |
| `salary_topcoded` | `SALARY` is 827,838 or 976,934 | Every salary above \$500k in the public file was replaced with one of these two values |
| `earn_topcoded` | `EARN` is 1,290,356 or 1,563,491 | Same top-coding for total earnings |
| `hours_topcoded` | `HRSWK` = 96 | Codebook: "96 hours or more" |

`log_salary` is the natural log of `SALARY` for EDA, since salaries are heavily right-skewed. It's blank for zero salaries (log of 0 is undefined).

In [37]:
NUMERIC = ["SALARY", "EARN", "HRSWK", "WKSWK", "AGE"]
df[NUMERIC] = df[NUMERIC].apply(pd.to_numeric)

SALARY_TOPCODES = [827838, 976934]
EARN_TOPCODES = [1290356, 1563491]

df["salary_zero"] = df.SALARY == 0
df["salary_low"] = df.SALARY.between(1, 999)
df["salary_topcoded"] = df.SALARY.isin(SALARY_TOPCODES)
df["earn_topcoded"] = df.EARN.isin(EARN_TOPCODES)
df["hours_topcoded"] = df.HRSWK == 96
df["log_salary"] = np.log(df.SALARY.where(df.SALARY > 0))

FLAGS = ["salary_zero", "salary_low", "salary_topcoded", "earn_topcoded", "hours_topcoded"]
df[FLAGS].sum().rename("rows flagged")

/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/2059894087.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[NUMERIC] = df[NUMERIC].apply(pd.to_numeric)
/var/folders/ny/p5hb2xbx46g25_c_0lg5mhyc0000gn/T/ipykernel_41027/2

salary_zero        287
salary_low         212
salary_topcoded    421
earn_topcoded      416
hours_topcoded      53
Name: rows flagged, dtype: int64

## 7. Leakage: satisfaction columns are not predictors

The nine `SAT*` facets (satisfaction with advancement, benefits, salary, ...) and `JOBSATIS` itself measure the outcome. Overall satisfaction is largely a summary of the facets, so a model given them would score well without learning anything about person-job fit.

- **Keep them in the data** to validate results, e.g. check that a predicted fit on salary lines up with `SATSAL`.
- **Never use them as features.** `LEAKAGE_COLS` below lists them so modeling code can drop them in one step.

The `FAC*` importance items are fine to use: they ask what the person values in a job, not how satisfied they are with this one.

In [38]:
LEAKAGE_COLS = ["JOBSATIS"] + SAT_COLS

# How closely each facet tracks the target, for comparison with the FAC* items
pd.DataFrame(
    {
        "SAT* vs very_satisfied": df[SAT_COLS].corrwith(df.very_satisfied).values,
        "FAC* vs very_satisfied": df[FAC_COLS].corrwith(df.very_satisfied).values,
    },
    index=FACTORS,
).round(2)

,SAT* vs very_satisfied,FAC* vs very_satisfied
ADV,0.47,-0.02
BEN,0.33,-0.01
CHAL,0.46,0.16
IND,0.40,0.15
LOC,0.27,0.07
RESP,0.47,0.15
SAL,0.43,-0.01
SEC,0.34,0.05
SOC,0.37,0.12


## 8. Missingness

NSCG imputes item nonresponse before releasing the public file, so the survey columns are complete apart from the codes blanked in step 2. The real gaps are on the O*NET side:

- **Work Values** are blank where O*NET never rated the occupation. Per the decision for this ticket, they stay blank and flagged (`wv_status`, `has_work_values`, `partial_substantive`) rather than being filled from older O*NET releases. The biggest gap is Software developers (110590), with no Work Values at all.
- **Job Zone** has fewer gaps, since O*NET covers more occupations there.
- **785000 Other occupations** has no O*NET match, so all its O*NET columns are blank.

The table lists every column with at least one blank.

In [39]:
# log_salary is blank by design for the zero salaries flagged in step 6
missing = df.isna()
summary = pd.DataFrame(
    {
        "n_missing": missing.sum(),
        "pct": (missing.mean() * 100).round(1),
        "weighted_pct": (missing.mul(df.WTSURVY, axis=0).sum() / df.WTSURVY.sum() * 100).round(1),
    }
)
summary[summary.n_missing > 0].sort_values("n_missing", ascending=False)

,n_missing,pct,weighted_pct
Achievement,4955,6.2,3.9
Independence,4955,6.2,3.9
Recognition,4955,6.2,3.9
Relationships,4955,6.2,3.9
Support,4955,6.2,3.9
Working Conditions,4955,6.2,3.9
job_zone,886,1.1,0.8
INDCODE,499,0.6,0.7
log_salary,287,0.4,0.5
EARN,99,0.1,0.2


## 9. Save the cleaned dataset

Written to `data/nscg2023_onet_clean.csv` (gitignored; rebuild by running this notebook). CSV doesn't store pandas dtypes, so the labeled categories from step 5c come back as plain text when the file is reloaded.

In [40]:
out = DATA / "nscg2023_onet_clean.csv"
df.to_csv(out, index=False)
print(f"Wrote {out}: {df.shape[0]} rows x {df.shape[1]} columns")

Wrote ../../data/nscg2023_onet_clean.csv: 79800 rows x 71 columns
